# 01 — Database Setup & Knowledge Base

This notebook initialises the UDA-Hub SQLite database, seeds sample tenants/users/tickets,
and loads the knowledge base used by the retrieval agent.

**Schema**

- `Account` — tenant/billing customer
- `User` — end customer
- `Ticket` — a support case
- `TicketMetadata` — channel, urgency, classification, sentiment, ...
- `TicketMessage` — append-only conversation log
- `Knowledge` — support articles

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # so `uda_hub` is importable from notebooks/

from uda_hub import db, seed
from uda_hub.config import settings
settings

## 1. Initialise (or reset) the database

In [ ]:
counts = seed.seed_all(reset=True)
counts

## 2. Inspect the schema

Confirm every required table exists and is populated.

In [ ]:
from tabulate import tabulate
rows = db.fetch_all(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
)
print(tabulate(rows, headers='keys'))

In [ ]:
for tbl in ['Account','User','Ticket','TicketMetadata','TicketMessage','Knowledge']:
    n = db.fetch_one(f'SELECT COUNT(*) AS n FROM {tbl}')['n']
    print(f'{tbl:<16} {n}')

## 3. Sample rows

In [ ]:
print(tabulate(db.fetch_all('SELECT account_id, name, plan, status FROM Account'), headers='keys'))

In [ ]:
print(tabulate(db.fetch_all('SELECT user_id, account_id, email, full_name FROM User'), headers='keys'))

In [ ]:
print(tabulate(
    db.fetch_all('SELECT ticket_id, user_id, subject, status FROM Ticket'),
    headers='keys', maxcolwidths=[None,None,40,None]
))

## 4. Knowledge base

In [ ]:
kb = db.fetch_all('SELECT category, COUNT(*) AS n FROM Knowledge GROUP BY category ORDER BY category')
print(tabulate(kb, headers='keys'))
total = db.fetch_one('SELECT COUNT(*) AS n FROM Knowledge')['n']
print(f'\nTotal articles: {total}  (rubric requires \u226514)')

In [ ]:
for row in db.fetch_all('SELECT article_id, title, category FROM Knowledge ORDER BY article_id'):
    print(f"{row['article_id']}  [{row['category']:<10}] {row['title']}")

## 5. Spot-check a single article

In [ ]:
art = db.fetch_one('SELECT * FROM Knowledge WHERE article_id=?', ('kb_010',))
print(art['title']); print('-'*60); print(art['body'])

## 6. Conversation history is preserved

Returning customers should see prior interactions retrieved later by the agents.

In [ ]:
print(tabulate(
    db.fetch_all("""
        SELECT t.ticket_id, t.subject, m.role, m.content
          FROM Ticket t JOIN TicketMessage m USING(ticket_id)
         WHERE t.user_id='usr_002'
         ORDER BY m.message_id
    """),
    headers='keys', maxcolwidths=[None,30,None,60]
))

---

Database is ready. Continue to `02_end_to_end_demo.ipynb` to run the multi-agent workflow.